# 🏠 Домашнее задание №1 — Решаем математику

## Цель

Выбить 100\100 по датасету

## Описание

Вам дан датасет с математическими вопросами (уравнения, арифметика, сравнения и др.).  
Каждый вопрос имеет:
- `question_id` — уникальный идентификатор
- `question` — текст задачи  
- `answer` — правильный ответ


**Ваша задача:**
1. Решить задачи с помощью LLM
2. Написать LLM flow, который для каждого вопроса:
   - отправляет вопрос модели
   - получаете результат
Решение может быть с помощью SGR, function_call. Решайте как считаете правильным. Главное результат

4. Прогнать на всём датасете и получить итоговый **score**

## Критерии оценки

- **Accuracy** — доля правильных ответов (совпадение значения)

---

> 💡 Вы можете использовать **любого провайдера** и **любую модель** с поддержкой 

---

## 📊 Оценка ДЗ (10-балльная шкала)

| Критерий | Баллы |
|---|---|
| **Промпт** (применены минимум 2 техники из лекции) | **1** |
| **Accuracy** ниже 85% | **0** |
| **Accuracy** 85–95% | **3** |
| **Accuracy** 95–99% | **4** |
| **Accuracy** 100% | **5** |
| Применение **SGR** и/или **function call** | **+2** |
| Дешёвая модель (< 20 руб / 1M токенов input И output) | **+2** |



#### Примеры дешёвых моделей: `Ministral 3 14B 2512`, `gpt-oss-20b`, `Qwen2.5-VL 7B Instruct`


### ⚠️ Штрафы

| Нарушение | Штраф |
|---|---|
| Модель дороже 200 руб / 1M токенов (input) | **−2** |
| Описание проблемных кейсов в промпте | **−2** |
| Нет retry в случае ошибки при запросе по API | **−1** |


#### "Описание проблемных кейсов в промпте" Это например указать в промпте что для вопроса 'Solve -34*v + 232*v + 52351 = 48985 for v.	' ответ -17

#### За модели средней стоимости (20–200 руб / 1M токенов) штрафов нет.

## 0. Настройка окружения

In [1]:
import warnings

warnings.filterwarnings("ignore")


import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists():
            return p
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.token_manager import GigaTokenManager
from src.config import settings
from src.utils import Answer, validate_answers

token_manager = GigaTokenManager()

In [2]:
%pip install --upgrade openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 2.3 MB/s  0:00:01 eta 0:00:01
  Attempting uninstall: openai
    Found existing installation: openai 3.52.0
    Uninstalling openai-3.52.0:
      Successfully uninstalled openai-3.52.0

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Загрузка датасета

Датасет содержит математические вопросы разных типов. Загрузим его и посмотрим на структуру.

In [2]:
import pandas as pd 
dataset = pd.read_excel(PROJECT_ROOT / "lectures" / "lecture_1" / "hw_1_dataset.xlsx")

In [3]:
dataset

,question_id,question,answer
0,4d637ecc-d8a8-4ef5-9467-259f51ebe692,Solve -282*d + 929 - 178 = -1223 for d.,7
1,27534870-1f8c-41e7-91c8-61a815c00392,Solve 49*l + 45*l - 125 - 63 = 0 for l.,2
2,cb319da5-2d1e-4814-a665-686521c56c02,Solve -64*t + 1387 - 848 + 933 = 0 for t.,23
3,0f9540f4-91cb-4d58-9e91-93931e4a8aeb,Solve 75*g = 192*g - 71*g - 79*g - 264 for g.,-8
4,d05be1f2-eb77-4175-915f-68c3269983ab,Solve -34*v + 232*v + 52351 = 48985 for v.,-17
...,...,...,...
106,d3bbfb3c-1674-4d6c-b883-21e699417455,"Sort 53, 279, 2, 1, 5, 4, -5.","-5, 1, 2, 4, 5, 53, 279"
107,f4c87be8-280e-4f46-90f6-4468641ec58e,"Sort 2, -0.2, -198, -1860.","-1860, -198, -0.2, 2"
108,1585d0d6-0fa0-46b9-9c05-3e91d96994ef,"Sort 5, -175, -4, -7, -30.","-175, -30, -7, -4, 5"
109,36033990-219d-4065-a2a5-08fa1cb8e7e4,"Sort -2, 3936, 15/19 in ascending order.","-2, 15/19, 3936"


In [3]:
print(f"Всего вопросов: {len(dataset)}")

Всего вопросов: 111


## 2. Настройка LLM-клиента

Создайте клиент для работы с LLM. Можно использовать любого провайдера

> ⚠️ Убедитесь, что выбранная модель поддерживает **function calling** (Если будете использовать function calling).

In [4]:
from openai import OpenAI

client = OpenAI(
  base_url="https://polza.ai/api/v1",
  api_key="pza_zFqtG7NKtfOYsqmlzceGmAle5OPr1ppY",
)

completion = client.chat.completions.create(
  model="openai/gpt-4o",
  messages=[{
    "role": "user",
    "content": "Что думаешь об этой жизни?"
  }]
)

print(completion.choices[0].message.content)

Жизнь — это сложный и многогранный опыт, который каждый человек воспринимает по-своему. Для одних это возможность для самовыражения и достижения целей, для других — источник испытаний и уроков. Жизнь полна радостей и трудностей, и в ней важно находить баланс между внешними обстоятельствами и внутренним миром. Все зависит от личных ценностей, ожиданий и восприятия, которые формируют уникальный путь каждого.


## 3. Обработка одного вопроса

Реализуйте функцию `ask_question()`, которая:

1. Отправляет вопрос модели 
2. Проверяет, вернула ли модель `function_call` (если решаете использовать `function_call`)
3. Если да — парсит аргументы, вызывает нужный function, возвращает результат модели
4. Получает финальный ответ


In [20]:

import os
import re
import sympy as sp

from decimal import Decimal, getcontext, ROUND_HALF_UP
from openai import OpenAI
from dotenv import load_dotenv


getcontext().prec = 50




def normalize_value(value):

    if isinstance(value, (list, tuple)):
        return ", ".join(str(normalize_value(x)) for x in value)

    if isinstance(value, dict):
        if len(value) == 1:
            return normalize_value(next(iter(value.values())))
        return str(value)

    if isinstance(value, sp.Integer):
        return int(value)

    if isinstance(value, sp.Rational):
        if value.q == 1:
            return int(value)
        return str(value)

    if isinstance(value, sp.Float):
        return str(value)

    if isinstance(value, Decimal):
        return format(value, "f")

    if isinstance(value, float):
        decimal_value = Decimal(str(value))
        if decimal_value == decimal_value.to_integral_value():
            return int(decimal_value)
        return format(decimal_value, "f")

    return value


def ask_question(problem):

    prompt = f"""
Ты — математический Python-агент.
Твоя задача — решить математическую задачу,
преобразовав её в исполняемый Python-код.

ОБЩИЙ АЛГОРИТМ:

1. Внимательно проанализируй условие.
2. Определи тип задачи.
3. Выбери подходящий алгоритм решения.
4. Напиши Python-код.
5. Выполни вычисления с помощью Python.
6. Проверь результат.
7. Сохрани только итоговый ответ в переменную answer.

МАТЕМАТИЧЕСКИЕ ВЫЧИСЛЕНИЯ:

- Используй SymPy (sp) для алгебраических задач.
- Используй sp.solve() для решения уравнений.
- Используй sp.simplify() для упрощения выражений.
- Для точных дробей используй sp.Rational().
- Не используй float для точных десятичных вычислений.
- Для десятичных вычислений используй Decimal.
- Не округляй числа без указания в условии.
- Не теряй знаки отрицательных чисел.
- Проверяй арифметические операции.
- Не используй [0] без проверки наличия элементов.

ДЕСЯТИЧНЫЕ ЧИСЛА:

- Используй Decimal("число"), а не float.
- Сохраняй требуемое количество знаков после запятой.
- Если требуется округление, используй quantize().
- Используй ROUND_HALF_UP для обычного математического округления.
- Не изменяй точность, если она указана в условии.

УРАВНЕНИЯ И СИСТЕМЫ:

- Определяй, какую именно переменную требуется найти.
- Для нескольких переменных используй sp.solve().
- Проверяй тип результата sp.solve().
- Если результат является словарём, извлекай значение нужной переменной.
- Не возвращай весь словарь вместо одного числа.
- Проверяй решение подстановкой в исходное уравнение.

ПОСЛЕДОВАТЕЛЬНОСТИ:

- Определи закономерность последовательности.
- Проверь разности, отношения и другие возможные закономерности.
- Вычисли следующий элемент.
- Сохрани только требуемое число в answer.

СОРТИРОВКА:

- Извлеки все числа из условия.
- Учитывай отрицательные числа и десятичные дроби.
- Используй sorted() для сортировки по возрастанию.
- Не теряй и не изменяй исходные значения.
- Выведи числа через запятую.
- Не используй квадратные скобки.

СИСТЕМЫ СЧИСЛЕНИЯ:

- Учитывай указанное основание системы счисления.
- Правильно обрабатывай отрицательные числа.
- Используй алгоритм перевода между системами счисления.
- Не интерпретируй цифры как обычное десятичное число,
  если задача требует другую систему счисления.

ФОРМАТ ОТВЕТА:

- Возвращай исключительно исполняемый Python-код.
- Не используй Markdown и блоки кода.
- Всегда создавай переменную answer.
- В answer должен находиться только итоговый результат.
- Не добавляй пояснения к ответу.
- Не используй input().
- Не используй несуществующие переменные.
- Не возвращай промежуточные вычисления.

Задача:
{problem}
"""

    try:

        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Ты математический Python-агент. "
                        "Решай задачи через исполняемый Python-код. "
                        "Возвращай только код, создающий переменную answer."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        code = response.choices[0].message.content.strip()

        
        code = re.sub(
            r"^```(?:python)?\s*",
            "",
            code,
            flags=re.IGNORECASE
        )
        code = re.sub(r"\s*```$", "", code)

      
        namespace = {
            "sp": sp,
            "Decimal": Decimal,
            "getcontext": getcontext,
            "ROUND_HALF_UP": ROUND_HALF_UP
        }

        
        exec(code, namespace, namespace)

        
        if "answer" not in namespace:
            print("Модель не создала переменную answer")
            return None

        answer = namespace["answer"]

        if answer is None:
            return None

        
        answer = normalize_value(answer)

        return str(answer).strip()

    except Exception as e:
        print(f"Ошибка при решении задачи: {e}")
        return None


### Проверка на одном вопросе

Прежде чем запускать на всём датасете, проверьте агента на одном примере.

In [22]:
# Тест на одном примере
test_row = dataset.iloc[0]
print(f"Вопрос:   {test_row['question']}")
print(f"Ожидаемо: {test_row['answer']}")

answer = ask_question(test_row["question"])
print(f"Ответ LLM: {answer}")

if answer is not None:
    print(f"Совпадает: {answer == test_row['answer']}")

Вопрос:   Solve -282*d + 929 - 178 = -1223 for d.
Ожидаемо: 7
Ответ LLM: 7
Совпадает: True


## 4. Прогон по всему датасету

Запустите агента на всех вопросах из датасета. Для каждого вопроса:
1. Вызовите `ask_question()` для получения ответа
2. Приведите ответ к нужному типу через `parse_answer()`
3. Сформируйте объект `Answer(question_id=..., answer=...)`

> ⏱️ Этот шаг может занять время — в зависимости от провайдера и количества вопросов.  
> При необходимости добавьте `time.sleep()` между запросами для соблюдения rate limits.

In [23]:
def parse_answer(answer):
    
    if answer is None:
        return None

    answer = str(answer).strip()

    return answer

In [24]:
from tqdm.auto import tqdm

answers: list[Answer] = []
failed: list[dict] = []

for idx, row in tqdm(dataset.iterrows(), total=len(dataset), desc="Обработка вопросов"):
    question_id = row["question_id"]
    question = row["question"]

    # Получаем ответ от агента
    answer = ask_question(question)

    if answer is None:
        failed.append({"index": idx, "question_id": question_id, "reason": "no_answer"})
        continue

    try:
        answers.append(Answer(question_id=question_id, answer=answer))
    except Exception as e:
        failed.append({
            "index": idx,
            "question_id": question_id,
            "reason": f"parse_error: {e}",
            "answer": answer,
        })

    
print(f"\nГотово!")
print(f"Ответов получено: {len(answers)} из {len(dataset)}")
print(f"Ошибок: {len(failed)}")

Обработка вопросов:  66%|██████▌   | 73/111 [04:08<02:33,  4.03s/it]

Ошибка при решении задачи: module 'sympy' has no attribute 'base_repr'


Обработка вопросов: 100%|██████████| 111/111 [06:02<00:00,  3.27s/it]


Готово!
Ответов получено: 110 из 111
Ошибок: 1


## 5. Валидация и подсчёт score

Используем функцию `validate_answers()` из `src.utils` для проверки ответов.  
Валидация проверяет:
- **Значение** — ответ должен **точно совпадать** с эталоном (сравнение по строковому представлению)

### Формула score

$$\text{accuracy} = \frac{\text{правильных ответов}}{\text{всего вопросов в датасете}}$$

In [25]:
# Валидация ответов
result = validate_answers(answers, dataset)

total_questions = len(dataset)
total_answered = len(answers)
total_correct = total_answered - len(result["errors"])

# Score считается от общего числа вопросов в датасете
accuracy = total_correct / total_questions if total_questions > 0 else 0

print("=" * 50)
print(f"📊 РЕЗУЛЬТАТЫ")
print("=" * 50)
print(f"Всего вопросов в датасете: {total_questions}")
print(f"Ответов отправлено:        {total_answered}")
print(f"Не удалось ответить:       {len(failed)}")
print(f"Ошибок валидации:          {len(result['errors'])}")
print(f"Правильных ответов:        {total_correct}")
print(f"")
print(f"🎯 Accuracy: {accuracy:.2%} ({total_correct}/{total_questions})")
print("=" * 50)

📊 РЕЗУЛЬТАТЫ
Всего вопросов в датасете: 111
Ответов отправлено:        110
Не удалось ответить:       1
Ошибок валидации:          9
Правильных ответов:        101

🎯 Accuracy: 90.99% (101/111)


In [26]:
# Детализация ошибок (первые 10)
if result["errors"]:
    print("Примеры ошибок:")
    for err in result["errors"][:10]:
        qid = err["question_id"]
        reason = err["reason"]
        row = dataset[dataset["question_id"] == qid]
        if not row.empty:
            q = row.iloc[0]["question"]
            print(f"\n  [{reason}] {q[:80]}...")
            if reason == "value_mismatch":
                print(f"    Ожидалось: {err['expected']}")
                print(f"    Получено:  {err['actual']}")
else:
    print("✅ Все ответы корректны!")

Примеры ошибок:

  [value_mismatch] Let q = -3 + -3. Let o be (16/q)/(1 - 5/3). Let k be o - ((-4 - -7) + -1). Solve...
    Ожидалось: -3
    Получено:  -3.00000000000000

  [value_mismatch] What is next in -8807, -9386, -9979, -10592, -11231, -11902, -12611, -13364?...
    Ожидалось: -14167
    Получено:  -14117

  [value_mismatch] What is the difference between -221017 and -1429.06? Answer with 2 decimal place...
    Ожидалось: 219587.94
    Получено:  -219587.94

  [value_mismatch] In base 11, what is 1219 - -4778?...
    Ожидалось: 5996
    Получено:  7849

  [value_mismatch] In base 7, what is 613213 - -105?...
    Ожидалось: 613321
    Получено:  104434

  [value_mismatch] In base 8, what is -300333 - -62?...
    Ожидалось: -300251
    Получено:  -98473

  [value_mismatch] In base 6, what is -1 - -23004345?...
    Ожидалось: 23004344
    Получено:  700840

  [value_mismatch] In base 7, what is -1523 - -45464?...
    Ожидалось: 43641
    Получено:  10956

  [value_mismatch] In bas

In [27]:
print(dataset.columns.tolist())

['question_id', 'question', 'answer']


In [29]:
def get_answer_type(answer):
    parsed = parse_answer(answer)

    if isinstance(parsed, bool):
        return "bool"
    elif isinstance(parsed, int):
        return "int"
    elif isinstance(parsed, float):
        return "float"
    elif isinstance(parsed, list):
        return "list"
    else:
        return "str"


dataset["answer_type"] = dataset["answer"].apply(get_answer_type)

In [30]:
# Accuracy по типам ответов
error_qids = {e["question_id"] for e in result["errors"]}
failed_qids = {f["question_id"] for f in failed}

print("Accuracy по типам ответов:")
print("-" * 40)
for atype in ["int", "float", "bool", "list", "str"]:
    subset = dataset[dataset["answer_type"] == atype]
    total = len(subset)
    wrong = sum(1 for _, r in subset.iterrows() 
                if str(r["question_id"]) in error_qids or str(r["question_id"]) in failed_qids)
    correct = total - wrong
    acc = correct / total if total > 0 else 0
    print(f"  {atype:>5}: {acc:.0%}  ({correct}/{total})")

Accuracy по типам ответов:
----------------------------------------
    int: 0%  (0/0)
  float: 0%  (0/0)
   bool: 0%  (0/0)
   list: 0%  (0/0)
    str: 91%  (101/111)


## 📝 Что нужно сдать

1. Этот ноутбук с **заполненным кодом** (все `TODO` реализованы)
2. Все ячейки должны быть **выполнены** — с выводом результатов
3. В выводе должен быть виден **итоговый score**

---

## 💡 Советы

- **Начните с малого**: сначала отладьте работу на единичных запросах или на малом датасете
- **Промпт имеет значение**: хороший `SYSTEM_PROMPT` сильно влияет на качество ответов
- **Обрабатывайте ошибки**: LLM иногда возвращает неожиданные ответы — ваш код не должен падать
---

GLHF 🚀